# LumenY 10 — System 1: Trend Continuation in Liquid Hours

**Thesis:** When all timeframes agree on direction, price is positioned correctly relative to structure, and microstructure confirms real flow — enter in the direction of the trend during liquid sessions.

**No training. Pure rules. Full history valid.**

| Layer | Feature | Condition | Rationale |
|-------|---------|-----------|----------|
| Geometric | `slope_agree_3_24` | = 1 | All timeframes aligned |
| Geometric | `slope_close_24h` | > 0 long / < 0 short | 24h slope confirms direction |
| Geometric | `range_pos_24h` | >0.6 long / <0.4 short | Price in upper/lower range |
| Microstructure | `hurst_6h` | > 0.55 | Trending not mean-reverting |
| Microstructure | `kyle_lambda` | > `kyle_lambda_ma12` | Informed flow above average |
| Microstructure | `entropy_norm` | < 0.85 | Directional not noisy |
| Session | `is_london` or `is_ny` | = 1 | Liquid hours only |

**Exit:** Fixed hold of 4H (tunable)

**Spread:** 2.8 pips average across pairs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

FEATURES_6_DIR = Path('../backend/data/features_6')
FEATURES_8_DIR = Path('../backend/data/features_8')
PROCESSED_DIR  = Path('../backend/data/processed')

AVG_SPREAD_PIPS = 2.8
HOLD_H = 4

PIP_SIZE = {
    'EURUSD':0.0001,'GBPUSD':0.0001,'AUDUSD':0.0001,'NZDUSD':0.0001,
    'USDCAD':0.0001,'USDCHF':0.0001,'USDJPY':0.01,
    'EURJPY':0.01,'GBPJPY':0.01,'AUDJPY':0.01,'CADJPY':0.01,
    'CHFJPY':0.01,'EURAUD':0.0001,'EURGBP':0.0001,'AUDNZD':0.0001,
}
# Max plausible 1H move — for data quality filter
MAX_1H_MOVE = {
    'EURUSD':150,'GBPUSD':200,'AUDUSD':120,'NZDUSD':100,'USDCAD':150,
    'USDCHF':150,'USDJPY':300,'EURJPY':300,'GBPJPY':350,'AUDJPY':250,
    'CADJPY':250,'CHFJPY':250,'EURAUD':150,'EURGBP':100,'AUDNZD':80,
}

PAIRS = list(PIP_SIZE.keys())
print('Setup complete.')


## 1. Load features_6 (microstructure) + features_8 (geometric) + 1H closes

In [ ]:
# Load features_6 (microstructure)
dfs6 = []
for f in sorted(FEATURES_6_DIR.glob('*_features.parquet')):
    dfs6.append(pd.read_parquet(f))
df6 = pd.concat(dfs6).sort_index()
del dfs6
print(f'features_6: {df6.shape}')

# Load features_8 (geometric)
dfs8 = []
for f in sorted(FEATURES_8_DIR.glob('*_geometric.parquet')):
    pair = f.stem.replace('_geometric','')
    tmp = pd.read_parquet(f)
    tmp['pair'] = pair
    dfs8.append(tmp)
df8 = pd.concat(dfs8).sort_index()
del dfs8
print(f'features_8: {df8.shape}')

# Merge on (datetime, pair)
df6r = df6.reset_index()
df8r = df8.reset_index().drop(columns=[c for c in df8.columns if c in df6.columns and c != 'pair'], errors='ignore')
idx_col = df6r.columns[0]
df = pd.merge(df6r, df8r, on=[idx_col, 'pair'], how='inner')
df = df.set_index(idx_col).sort_index()
print(f'merged: {df.shape}')

# Load 1H closes for forward return computation
closes = {}
for pair in PAIRS:
    df1h = pd.read_parquet(PROCESSED_DIR / f'{pair}_1H.parquet')
    if 'datetime' in df1h.columns: df1h = df1h.set_index('datetime')
    df1h.index = pd.to_datetime(df1h.index)
    closes[pair] = df1h['close']
print(f'Loaded closes for {len(closes)} pairs')


## 2. Compute forward returns + data quality filter

In [ ]:
rows = []
for pair in PAIRS:
    pip = PIP_SIZE[pair]
    max_move = MAX_1H_MOVE[pair]  # in pips
    df_p = df[df['pair'] == pair].copy()
    close = closes[pair].reindex(df_p.index)

    # Forward return for HOLD_H bars (simple price diff in pips)
    df_p['fwd_ret_pips'] = (close.shift(-HOLD_H) - close) / pip

    # Data quality: exclude bad ticks and holidays
    ret_1h = ((close.shift(-1) - close) / pip).abs()
    holiday = ~(
        ((df_p.index.month==12) & (df_p.index.day.isin([24,25,26,31]))) |
        ((df_p.index.month==1)  & (df_p.index.day.isin([1,2])))
    )
    df_p['clean'] = (ret_1h <= max_move) & holiday
    rows.append(df_p)

df_all = pd.concat(rows).sort_index()
print(f'Full dataset: {df_all.shape}')
print(f'Date range: {df_all.index.min().date()} -> {df_all.index.max().date()}')
print(f'Clean bars: {df_all["clean"].sum():,} / {len(df_all):,}')


## 3. System 1 rules

In [ ]:
def apply_system1(df, hold_h=4, spread=2.8):
    """
    System 1: Trend Continuation in Liquid Hours
    Returns DataFrame with signal column: +1 long, -1 short, 0 no trade
    """
    d = df.copy()

    # ── LONG conditions ──────────────────────────────────────────────
    long_geo = (
        (d['slope_agree_3_24'] == 1) &       # all timeframes up
        (d['slope_close_24h'] > 0) &          # 24h slope positive (trending up)
        (d['range_pos_24h'] > 0.6)            # price in upper part of 24h range
    )
    long_micro = (
        (d['hurst_6h'] > 0.55) &             # trending regime
        (d['kyle_lambda'] > d['kyle_lambda_ma12']) &  # informed flow above avg
        (d['entropy_norm'] < 0.85)            # directional not noisy
    )

    # ── SHORT conditions ─────────────────────────────────────────────
    short_geo = (
        (d['slope_agree_3_24'] == 1) &        # all timeframes aligned
        (d['slope_close_24h'] < 0) &           # 24h slope negative (trending down)
        (d['range_pos_24h'] < 0.4)            # price in lower part of 24h range
    )
    short_micro = (
        (d['hurst_6h'] > 0.55) &
        (d['kyle_lambda'] > d['kyle_lambda_ma12']) &
        (d['entropy_norm'] < 0.85)
    )

    # ── Session filter ───────────────────────────────────────────────
    liquid = (d['is_london'] == 1) | (d['is_ny'] == 1)

    # ── Combine ──────────────────────────────────────────────────────
    d['signal'] = 0
    d.loc[long_geo  & long_micro  & liquid & d['clean'], 'signal'] =  1
    d.loc[short_geo & short_micro & liquid & d['clean'], 'signal'] = -1

    # ── PnL ──────────────────────────────────────────────────────────
    d['pnl_pips'] = np.where(
        d['signal'] != 0,
        d['signal'] * d['fwd_ret_pips'] - spread,
        np.nan
    )
    return d

df_result = apply_system1(df_all)
trades = df_result[df_result['signal'] != 0].copy()
print(f'Total signals: {len(trades):,}')
print(f'  Long:  {(trades["signal"]==1).sum():,}')
print(f'  Short: {(trades["signal"]==-1).sum():,}')
print(f'Trades/month: {len(trades) / ((df_all.index.max()-df_all.index.min()).days/30):.1f}')


## 4. Results

In [ ]:
pnl = trades['pnl_pips'].dropna()
wins = pnl[pnl > 0]
losses = pnl[pnl <= 0]

print('=' * 55)
print('SYSTEM 1 — FULL HISTORY RESULTS')
print('=' * 55)
print(f'Period:        {df_all.index.min().date()} -> {df_all.index.max().date()}')
print(f'Trades:        {len(pnl):,}')
print(f'Trades/month:  {len(pnl)/((df_all.index.max()-df_all.index.min()).days/30):.1f}')
print(f'Win rate:      {(pnl>0).mean():.1%}')
print(f'Avg win:       {wins.mean():.1f} pips')
print(f'Avg loss:      {losses.mean():.1f} pips')
print(f'Median PnL:    {pnl.median():.1f} pips')
print(f'EV/trade:      {pnl.mean():.1f} pips')
print(f'Profit factor: {wins.sum()/abs(losses.sum()):.2f}')
print(f'Total PnL:     {pnl.sum():.0f} pips')
sharpe = (pnl.mean()/pnl.std()) * np.sqrt(252*24/HOLD_H) if pnl.std()>0 else 0
print(f'Sharpe:        {sharpe:.2f}')

print()
# Last 18 months
cutoff_18m = df_all.index.max() - pd.DateOffset(months=18)
recent = trades[trades.index >= cutoff_18m]['pnl_pips'].dropna()
if len(recent) > 10:
    print(f'--- Last 18 months ({cutoff_18m.date()} -> {df_all.index.max().date()}) ---')
    print(f'Trades:        {len(recent):,}')
    print(f'Win rate:      {(recent>0).mean():.1%}')
    print(f'EV/trade:      {recent.mean():.1f} pips')
    print(f'Total PnL:     {recent.sum():.0f} pips')
    sharpe_r = (recent.mean()/recent.std()) * np.sqrt(252*24/HOLD_H) if recent.std()>0 else 0
    print(f'Sharpe:        {sharpe_r:.2f}')

print()
print('Per-pair breakdown (full history):')
print(f'{"Pair":<10} {"Trades":>8} {"WinRate":>9} {"EV/trade":>10} {"Total":>10} {"Sharpe":>8}')
print('-' * 60)
for pair in sorted(trades['pair'].unique()):
    p = trades[trades['pair']==pair]['pnl_pips'].dropna()
    if len(p) < 5: continue
    sh = (p.mean()/p.std())*np.sqrt(252*24/HOLD_H) if p.std()>0 else 0
    flag = ' <<<' if p.mean() > 0 else ''
    print(f'{pair:<10} {len(p):>8,} {(p>0).mean():>8.1%} {p.mean():>10.1f} {p.sum():>10.0f} {sh:>8.2f}{flag}')


## 5. Hour distribution

In [ ]:
trades['hour'] = trades.index.hour
hourly = trades.groupby('hour')['pnl_pips'].agg(
    trades='count', win_rate=lambda x: (x>0).mean(), ev='mean', total='sum'
).reset_index()

print('Trade distribution by hour:')
print(f'{"Hour":>6} {"Trades":>8} {"WinRate":>9} {"EV/trade":>10} {"Total":>10}')
print('-' * 48)
for _, row in hourly.iterrows():
    flag = ' <<<' if row['ev'] > 0 else ''
    print(f'{int(row["hour"]):>6} {int(row["trades"]):>8,} {row["win_rate"]:>8.1%} {row["ev"]:>10.1f} {row["total"]:>10.0f}{flag}')
print(f'\nTrade count std across hours: {hourly["trades"].std():.1f} (lower = more uniform)')


## 6. Equity curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#080c14')
for ax in axes.flatten():
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

# Full history equity
cum = trades.sort_index()['pnl_pips'].cumsum()
axes[0,0].plot(cum.index, cum.values, color='#4fc3f7', linewidth=1.5)
axes[0,0].axhline(0, color='white', alpha=0.2)
axes[0,0].set_title(f'Full history — {len(trades):,} trades', color='white')
axes[0,0].set_ylabel('Cumulative pips', color='white')

# Per-pair
for pair in sorted(trades['pair'].unique()):
    pp = trades[trades['pair']==pair].sort_index()['pnl_pips'].cumsum()
    axes[0,1].plot(pp.index, pp.values, linewidth=0.8, alpha=0.6, label=pair)
axes[0,1].axhline(0, color='white', alpha=0.2)
axes[0,1].set_title('Per-pair equity', color='white')
axes[0,1].legend(fontsize=6, facecolor='#1a2332', labelcolor='white', ncol=3)

# Last 18 months
recent_trades = trades[trades.index >= cutoff_18m]
cum_r = recent_trades.sort_index()['pnl_pips'].cumsum()
axes[1,0].plot(cum_r.index, cum_r.values, color='#2ecc71', linewidth=1.5)
axes[1,0].axhline(0, color='white', alpha=0.2)
axes[1,0].set_title(f'Last 18 months — {len(recent_trades):,} trades', color='white')
axes[1,0].set_ylabel('Cumulative pips', color='white')

# Hour distribution
colors = ['#2ecc71' if v > 0 else '#ff4757' for v in hourly['ev']]
axes[1,1].bar(hourly['hour'], hourly['ev'], color=colors, alpha=0.8)
axes[1,1].axhline(0, color='white', alpha=0.3)
axes[1,1].set_title('EV/trade by hour (pips)', color='white')
axes[1,1].set_xlabel('Hour (UTC)', color='white')

plt.suptitle('System 1: Trend Continuation in Liquid Hours', color='white', fontsize=13)
plt.tight_layout()
plt.show()


## 7. Rule contribution analysis

Which conditions are doing the work? Remove one at a time and see what breaks.

In [ ]:
rules = {
    'Full system':          lambda d: d,
    'No slope_agree':       lambda d: d.assign(**{'slope_agree_3_24': 1}),  # always true
    'No slope_24h filter':  lambda d: d.assign(slope_close_24h=1e-6),  # always passes long side
    'No range_pos filter':  lambda d: d.assign(range_pos_24h=0.5),  # neutral
    'No hurst filter':      lambda d: d.assign(hurst_6h=0.6),       # always passes
    'No kyle filter':       lambda d: d.assign(kyle_lambda=lambda x: x['kyle_lambda_ma12']+1e-10),
    'No entropy filter':    lambda d: d.assign(entropy_norm=0.5),   # always passes
    'No session filter':    lambda d: d.assign(is_london=1, is_ny=1),
}

print('Rule ablation — what happens when we remove each condition:')
print(f'{"Variant":<25} {"Trades":>8} {"WinRate":>9} {"EV/trade":>10} {"Total":>10} {"Sharpe":>8}')
print('-' * 75)

for name, transform in rules.items():
    try:
        res = apply_system1(transform(df_all.copy()))
        t = res[res['signal'] != 0]['pnl_pips'].dropna()
        if len(t) < 5:
            print(f'{name:<25} insufficient trades')
            continue
        sh = (t.mean()/t.std())*np.sqrt(252*24/HOLD_H) if t.std()>0 else 0
        flag = ' <<<' if t.mean() > 0 else ''
        print(f'{name:<25} {len(t):>8,} {(t>0).mean():>8.1%} {t.mean():>10.1f} {t.sum():>10.0f} {sh:>8.2f}{flag}')
    except Exception as e:
        print(f'{name:<25} ERROR: {e}')
